# 🧠 NeuroLingua-Bridge — Colab Notebook
### Diagnosis of Autism Spectrum Disorder using LLMs and Multimodal Brain Connectivity Analysis

**Author:** Mr. Khalid EL KHAMLICHI

This notebook runs **top-to-bottom in Google Colab**. It implements the paper's
domain-generalization methodology (not a backbone swap of fMRI-LM):

1. **Multi-class site-adversarial GRL** — 17-way site discriminator + gradient reversal → `I(z;s)→0`.
2. **Dynamic latent prototypes** — site-balanced EMA anchor points in embedding space.
3. **Episodic Group-DRO** — site-as-task meta-learning, minimizes *worst-site* risk.

**Main model:** DeepSeek-Coder-1.3B · **Benchmarks:** GPT-2, ClinicalT5, Grok, Gemini.

---
### How to use in Colab
1. `Runtime → Change runtime type → GPU` (T4 is enough for a small run).
2. `Runtime → Run all`.
3. The first run downloads ABIDE-I (~2 GB) and the DeepSeek weights; later runs are cached.

> ⚠️ Set `QUICK_TEST = True` (cell 2) for a fast smoke run on a subset. Set it to
> `False` for the full experiment. All printed metrics in QUICK_TEST mode are not
> meaningful — they only confirm the pipeline executes.


## 0. Environment — GPU check & install

In [ ]:
# Check GPU
import subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "No GPU — set Runtime>GPU")


In [ ]:
# Install dependencies (Colab already has torch). ~1-2 min.
!pip -q install nilearn==0.10.4 transformers==4.44.2 accelerate sentencepiece networkx scikit-learn >/dev/null 2>&1
print("dependencies installed")


## 1. Configuration

In [ ]:
import os, math, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- master switch ----------
QUICK_TEST   = True       # True: subset + few epochs (smoke run). False: full experiment.
# -----------------------------------

# data / model dims
N_ROIS   = 200            # CC200 (native, no interpolation)
T_TARGET = 160
K_SITES  = 17             # ABIDE-I sites
D_ENC, D_VQ, K_VQ, PATCH = 256, 128, 8192, 32

# backbone selection: "deepseek" (main) | "gpt2" | "clinicalt5"
BACKBONE = "deepseek"
D_LLM = {"deepseek":2048, "gpt2":1024, "clinicalt5":1024}[BACKBONE]

# training budget
if QUICK_TEST:
    MAX_SUBJECTS, S1_EPOCHS, S3_EPOCHS, BATCH = 120, 2, 3, 4
else:
    MAX_SUBJECTS, S1_EPOCHS, S3_EPOCHS, BATCH = None, 15, 25, 8

print(f"device={DEVICE} | backbone={BACKBONE} (D_LLM={D_LLM}) | QUICK_TEST={QUICK_TEST}")


## 2. Download ABIDE-I (CC200) and preprocess

Native **CC200** derivative (200 ROIs, no interpolation), resample to 160 timepoints,
robust z-score, and build an **integer site index** per subject (needed by the
site-adversarial GRL and Group-DRO).


In [ ]:
from nilearn.datasets import fetch_abide_pcp
from nilearn.connectome import ConnectivityMeasure
from scipy.interpolate import interp1d
import pandas as pd
from tqdm.auto import tqdm

def load_abide_cc200(data_dir="/content/abide", max_subjects=None):
    ab = fetch_abide_pcp(data_dir=data_dir, pipeline="cpac",
                         band_pass_filtering=True, global_signal_regression=False,
                         derivatives=["rois_cc200"], quality_checked=True, verbose=0)
    pheno = pd.DataFrame(ab.phenotypic)
    dx    = pd.to_numeric(pheno["DX_GROUP"], errors="coerce").fillna(0).values
    site_raw = pheno.get("SITE_ID", pd.Series(["UNK"]*len(pheno))).values
    series = ab.rois_cc200
    idxs = range(len(series)) if max_subjects is None else range(min(max_subjects,len(series)))

    def resample(ts,T=T_TARGET):
        if ts.shape[0]==T: return ts
        f=interp1d(np.linspace(0,1,ts.shape[0]),ts,axis=0,kind="linear")
        return f(np.linspace(0,1,T)).astype(np.float32)
    def fit_rois(ts,N=N_ROIS):
        if ts.shape[1]==N: return ts.astype(np.float32)
        if ts.shape[1]>N:  return ts[:,:N].astype(np.float32)
        return np.pad(ts,((0,0),(0,N-ts.shape[1]))).astype(np.float32)
    def rz(ts):
        q75,q25=np.percentile(ts,75,0),np.percentile(ts,25,0)
        return ((ts-np.median(ts,0))/(q75-q25+1e-8)).astype(np.float32)

    TS,LBL,SITE=[],[],[]
    for i in tqdm(idxs, desc="preprocess"):
        a=np.asarray(series[i],dtype=np.float32)
        if a.ndim!=2 or min(a.shape)<10: continue
        if a.shape[0]<a.shape[1]: a=a.T            # -> (T, ROI)
        a=rz(fit_rois(resample(a)))
        TS.append(a); LBL.append(1 if int(dx[i])==1 else 0); SITE.append(site_raw[i])

    TS=np.stack(TS); LBL=np.array(LBL,np.int64)
    sites=sorted(pd.unique(SITE)); s2i={s:k for k,s in enumerate(sites)}
    SITE_IDX=np.array([s2i[s] for s in SITE],np.int64)
    print(f"{len(TS)} subjects | ASD={LBL.sum()} CTL={(LBL==0).sum()} | {len(sites)} sites")
    return TS, LBL, SITE_IDX, s2i, sites

TS, LBL, SITE_IDX, SITE2IDX, SITE_NAMES = load_abide_cc200(max_subjects=MAX_SUBJECTS)
K_SITES = len(SITE_NAMES)            # actual number present
print("K_SITES set to", K_SITES)


### 2b. FC-correlation text descriptors (supervision for SigLIP)

In [ ]:
def fc_corr(ts):
    cm=ConnectivityMeasure(kind="correlation",vectorize=False)
    fc=cm.fit_transform([ts])[0]; np.fill_diagonal(fc,0.0)
    return np.arctanh(np.clip(fc,-0.999,0.999)).astype(np.float32)

BLK=N_ROIS//8
NETS=[list(range(i*BLK,(i+1)*BLK)) for i in range(7)]+[list(range(7*BLK,N_ROIS))]
NAMES=["Visual","SomMot","DorsAttn","SalVent","Limbic","Cont","Default","Subcort"]
def descriptor(fc):
    pv={}
    for i,(a,ra) in enumerate(zip(NAMES,NETS)):
        for j,(b,rb) in enumerate(zip(NAMES,NETS)):
            if i<j: pv[f"{a}-{b}"]=float(fc[np.ix_(ra,rb)].mean())
    top=sorted(pv,key=pv.get,reverse=True)[:3]; bot=sorted(pv,key=pv.get)[:3]
    return (f"FC top=[{','.join(f'{k}({pv[k]:+.2f})' for k in top)}] "
            f"bot=[{','.join(f'{k}({pv[k]:+.2f})' for k in bot)}].")

print("building descriptors...")
DESCS=[descriptor(fc_corr(TS[i])) for i in tqdm(range(len(TS)))]
print("example:", DESCS[0])


## 3. Inter-site split + DataLoaders (yield ts, label, site)

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

def inter_site_split(lbl, site_idx, train_ratio=0.70):
    counts=pd.Series(site_idx).value_counts().sort_values(ascending=False)
    cum=counts.cumsum()/len(site_idx)
    big=counts[cum<=train_ratio].index.tolist(); small=counts[cum>train_ratio].index.tolist()
    if len(small)<2:
        n=max(2,len(counts)//4); small=counts.index[-n:].tolist(); big=counts.index[:-n].tolist()
    trf=np.where(np.isin(site_idx,big))[0]; te=np.where(np.isin(site_idx,small))[0]
    strat = lbl[trf] if len(np.unique(lbl[trf]))>1 else None
    tr,va=train_test_split(trf,test_size=0.15,stratify=strat,random_state=SEED)
    print(f"train={len(tr)} val={len(va)} test={len(te)} | train_sites={len(big)} test_sites={len(small)}")
    return tr,va,te

TR,VA,TE = inter_site_split(LBL, SITE_IDX)

class ABIDEDS(Dataset):
    def __init__(self, idx):
        self.idx=idx
    def __len__(self): return len(self.idx)
    def __getitem__(self,i):
        j=self.idx[i]
        return {"ts":torch.from_numpy(TS[j]),
                "label":torch.tensor(LBL[j]),
                "site":torch.tensor(SITE_IDX[j]),
                "desc_i":torch.tensor(j)}

def loader(idx, shuffle):
    return DataLoader(ABIDEDS(idx), batch_size=BATCH, shuffle=shuffle, drop_last=False)

train_loader=loader(TR,True); val_loader=loader(VA,False); test_loader=loader(TE,False)
print("loaders ready")


## 4. Text encoder (DeepSeek main) — descriptor → embedding

We freeze the LLM and use its last hidden state (mean-pooled) as the text anchor for the
SigLIP loss. Embeddings are cached per subject to avoid recomputation.


In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

def build_text_encoder(backbone):
    if backbone=="deepseek":
        name="deepseek-ai/deepseek-coder-1.3b-base"
        tok=AutoTokenizer.from_pretrained(name, trust_remote_code=True)
        mdl=AutoModelForCausalLM.from_pretrained(
            name, trust_remote_code=True,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            output_hidden_states=True)
        if tok.pad_token is None: tok.pad_token=tok.eos_token
    elif backbone=="gpt2":
        name="gpt2-medium"; tok=AutoTokenizer.from_pretrained(name)
        tok.pad_token=tok.eos_token; mdl=AutoModel.from_pretrained(name)
    elif backbone=="clinicalt5":
        from transformers import T5Tokenizer, T5EncoderModel
        try:
            name="luqh/ClinicalT5-large"; tok=T5Tokenizer.from_pretrained(name)
            mdl=T5EncoderModel.from_pretrained(name)
        except Exception:
            name="google/flan-t5-large"; tok=T5Tokenizer.from_pretrained(name)
            mdl=T5EncoderModel.from_pretrained(name)
    mdl=mdl.to(DEVICE).eval()
    for p in mdl.parameters(): p.requires_grad=False
    print("text encoder:", name)
    return tok, mdl

text_tok, text_mdl = build_text_encoder(BACKBONE)

@torch.no_grad()
def embed_text(strings, max_len=96):
    enc=text_tok(strings, return_tensors="pt", padding="max_length",
                 truncation=True, max_length=max_len).to(DEVICE)
    out=text_mdl(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
    h = out.hidden_states[-1] if hasattr(out,"hidden_states") and out.hidden_states is not None else out.last_hidden_state
    mask=enc["attention_mask"].unsqueeze(-1).float()
    emb=(h*mask).sum(1)/mask.sum(1).clamp(min=1)
    return emb.float()

# cache all descriptor embeddings once
print("caching text embeddings...")
TEXT_EMB=torch.zeros(len(DESCS), D_LLM)
for s in tqdm(range(0,len(DESCS),16)):
    chunk=DESCS[s:s+16]
    TEXT_EMB[s:s+len(chunk)]=embed_text(chunk).cpu()
def text_emb_fn(batch): return TEXT_EMB[batch["desc_i"]]
print("TEXT_EMB:", tuple(TEXT_EMB.shape))


## 5. Model — GRL, prototypes, tokenizer, loss weights

In [ ]:
# ---- Gradient Reversal ----
class GradReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx,x,lam): ctx.lam=lam; return x.view_as(x)
    @staticmethod
    def backward(ctx,g):    return g.neg()*ctx.lam, None
def grl_lambda(step,total,gamma=10.0):
    p=step/max(total-1,1); return 2.0/(1.0+math.exp(-gamma*p))-1.0

# ---- 17-way site discriminator ----
class SiteDiscriminator(nn.Module):
    def __init__(self,d_in=D_ENC,k=K_SITES):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(d_in,256),nn.LayerNorm(256),nn.ReLU(),
                               nn.Dropout(0.3),nn.Linear(256,128),nn.ReLU(),nn.Linear(128,k))
    def forward(self,z): return self.net(z)

# ---- dynamic latent prototypes (site-balanced EMA) ----
class DynamicPrototypes(nn.Module):
    def __init__(self,dim=D_ENC,n=2,eta=0.9,beta=0.5):
        super().__init__()
        self.register_buffer("mu",torch.zeros(n,dim))
        self.register_buffer("init",torch.zeros(1)); self.eta,self.beta,self.n=eta,beta,n
    @torch.no_grad()
    def update(self,z,y,site):
        new=self.mu.clone()
        for c in range(self.n):
            ps=[z[(y==c)&(site==k)].mean(0) for k in range(K_SITES) if ((y==c)&(site==k)).sum()>0]
            if ps: new[c]=torch.stack(ps).mean(0)
        if self.init.item()==0: self.mu.copy_(new); self.init.fill_(1.)
        else: self.mu.mul_(self.eta).add_(new,alpha=1-self.eta)
    def loss(self,z,y):
        d=torch.cdist(z,self.mu)
        own=d.gather(1,y.view(-1,1)).squeeze(1)
        other=d.clone(); other.scatter_(1,y.view(-1,1),float("inf"))
        return (own.pow(2)-self.beta*other.min(1).values.pow(2)).mean()

# ---- VQ + encoder + tokenizer ----
class NormEMAVQ(nn.Module):
    def __init__(self,K=K_VQ,D=D_VQ,decay=0.99):
        super().__init__(); self.K,self.D,self.decay=K,D,decay
        self.embed=nn.Embedding(K,D); nn.init.uniform_(self.embed.weight,-1/K,1/K)
        self.register_buffer("cnt",torch.ones(K)); self.register_buffer("w",self.embed.weight.data.clone())
    def forward(self,z):
        zf=F.normalize(z,dim=-1).reshape(-1,self.D); en=F.normalize(self.embed.weight,dim=-1)
        d=zf.pow(2).sum(1,True)+en.pow(2).sum(1)-2*zf@en.T; idx=d.argmin(1)
        zq=self.embed(idx).view(z.shape)
        if self.training:
            oh=F.one_hot(idx,self.K).float()
            self.cnt.mul_(self.decay).add_(oh.sum(0),alpha=1-self.decay)
            self.w.mul_(self.decay).add_(oh.T@zf,alpha=1-self.decay)
            self.embed.weight.data.copy_(F.normalize(self.w/(self.cnt.unsqueeze(1)+1e-5),dim=-1))
        return z+(zq-z).detach(), F.mse_loss(zq.detach(),z), idx

class fMRIEncoder(nn.Module):
    def __init__(self,n=N_ROIS,d=D_ENC,nh=8,nl=3,P=PATCH):
        super().__init__(); self.P=P
        self.roi=nn.Linear(n,d); self.t=nn.Linear(P,d); self.comb=nn.Linear(2*d,d)
        e=nn.TransformerEncoderLayer(d,nh,4*d,0.1,batch_first=True,norm_first=True)
        self.tf=nn.TransformerEncoder(e,nl); self.norm=nn.LayerNorm(d)
    def forward(self,x):
        B,T,N=x.shape; T2=T//self.P; xp=x[:,:T2*self.P,:].reshape(B,T2,self.P,N)
        z=self.comb(torch.cat([self.roi(xp.mean(2)),self.t(xp.mean(-1))],-1))
        return self.norm(self.tf(z))

class Tokenizer(nn.Module):
    def __init__(self,d_llm=D_LLM):
        super().__init__()
        self.encoder=fMRIEncoder()
        self.task=nn.Sequential(nn.Linear(D_ENC,D_ENC),nn.Tanh(),nn.Linear(D_ENC,D_VQ))
        self.vq=NormEMAVQ(); self.proj=nn.Sequential(nn.Linear(D_VQ,d_llm),nn.GELU())
        self.fmri_head=nn.Sequential(nn.Linear(D_ENC,256),nn.LayerNorm(256))
        self.text_head=nn.Sequential(nn.Linear(d_llm,256),nn.LayerNorm(256))
        self.site_disc=SiteDiscriminator(); self.protos=DynamicPrototypes()
        self.log_temp=nn.Parameter(torch.ones([])*math.log(1/0.07))
    def encode(self,x):
        zq,_,_=self.vq(self.task(self.encoder(x))); return self.proj(zq)

class LossWeights(nn.Module):
    def __init__(self,wq=0.5,ws=0.3,wg=0.2):
        super().__init__(); self.raw=nn.Parameter(torch.tensor([math.log(wq),math.log(ws),math.log(wg)]))
    @property
    def w(self): return F.softmax(self.raw,0)
    def forward(self,lq,ls,lg): w=self.w; return w[0]*lq+w[1]*ls+w[2]*lg

print("model defined")


## 6. Stage 1 — tokenizer training (recon + SigLIP + site-adversarial + prototypes)

In [ ]:
def siglip_loss(zf,zt,log_temp):
    zf=F.normalize(zf,dim=-1); zt=F.normalize(zt,dim=-1)
    logits=zf@zt.T*log_temp.exp(); labels=torch.arange(zf.size(0),device=zf.device)
    return 0.5*(F.cross_entropy(logits,labels)+F.cross_entropy(logits.T,labels))

def train_stage1(epochs=S1_EPOCHS):
    tok=Tokenizer().to(DEVICE).train()
    lw=LossWeights().to(DEVICE)
    opt=torch.optim.AdamW(list(tok.parameters())+list(lw.parameters()),lr=1e-4,weight_decay=1e-4)
    total=epochs*max(len(train_loader),1); step=0; whist=[]
    for ep in range(epochs):
        for b in train_loader:
            x=b["ts"].to(DEVICE); y=b["label"].to(DEVICE); site=b["site"].to(DEVICE)
            lam=grl_lambda(step,total)
            z=tok.encoder(x); zbar=z.mean(1)
            _,lc,_=tok.vq(tok.task(z))
            t_emb=text_emb_fn(b).to(DEVICE)
            l_sig=siglip_loss(tok.fmri_head(zbar),tok.text_head(t_emb),tok.log_temp)
            l_site=F.cross_entropy(tok.site_disc(GradReversal.apply(zbar,lam)),site)
            tok.protos.update(zbar.detach(),y,site)
            l_proto=tok.protos.loss(zbar,y)
            loss=lw(lc,l_sig,l_site)+0.3*l_proto
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(list(tok.parameters())+list(lw.parameters()),1.0)
            opt.step(); step+=1; whist.append(lw.w.detach().cpu().numpy())
        w=lw.w.detach().cpu().numpy().round(3)
        print(f"S1 ep{ep+1}/{epochs}  w(q,s,g)={w}  quant={lc.item():.3f} sig={l_sig.item():.3f} site={l_site.item():.3f}")
    return tok, lw, np.array(whist)

tok, lw, whist = train_stage1()


## 7. Stage 3 — ASD classifier with episodic Group-DRO (worst-site)

In [ ]:
class ASDClassifier(nn.Module):
    def __init__(self,tok,d_llm=D_LLM,hidden=512):
        super().__init__(); self.tok=tok
        for p in self.tok.parameters(): p.requires_grad=False
        self.attn=nn.Sequential(nn.Linear(d_llm,1),nn.Softmax(dim=1))
        self.feat=nn.Sequential(nn.Linear(d_llm,hidden),nn.LayerNorm(hidden),nn.GELU(),
                                nn.Dropout(0.35),nn.Linear(hidden,hidden//2),nn.GELU())
        self.head=nn.Linear(hidden//2,2)
    def forward(self,x):
        h=self.tok.encode(x); a=self.attn(h); return self.head(self.feat((h*a).sum(1)))

class GroupDRO:
    def __init__(self,k=K_SITES,lr_q=0.01): self.q=np.ones(k)/k; self.lr_q=lr_q
    def weight(self,psl):
        for k,l in psl.items(): self.q[k]*=math.exp(self.lr_q*l)
        self.q/=self.q.sum(); return self.q

def train_stage3(tok,epochs=S3_EPOCHS):
    clf=ASDClassifier(tok).to(DEVICE).train()
    opt=torch.optim.AdamW([p for p in clf.parameters() if p.requires_grad],lr=8e-4,weight_decay=1e-2)
    dro=GroupDRO()
    for ep in range(epochs):
        last=0.0
        for b in train_loader:
            x=b["ts"].to(DEVICE); y=b["label"].to(DEVICE); site=b["site"].numpy()
            per=F.cross_entropy(clf(x),y,reduction="none")
            psl={int(k):per[site==k].mean().item() for k in np.unique(site)}
            q=dro.weight(psl)
            w=torch.tensor([q[int(s)] for s in site],device=DEVICE,dtype=torch.float32)
            loss=(w*per).sum()/w.sum()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(clf.parameters(),0.8); opt.step(); last=loss.item()
        print(f"S3 ep{ep+1}/{epochs}  loss={last:.3f}  hardest_site={int(q.argmax())}")
    return clf

clf = train_stage3(tok)


## 8. Evaluation on held-out unseen sites (+ worst-site accuracy)

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                                   roc_auc_score, f1_score, recall_score)

@torch.no_grad()
def evaluate(clf, loader):
    clf.eval(); P,Y,S=[],[],[]
    for b in loader:
        prob=F.softmax(clf(b["ts"].to(DEVICE)),1)[:,1].cpu().numpy()
        P.append(prob); Y.append(b["label"].numpy()); S.append(b["site"].numpy())
    P=np.concatenate(P); Y=np.concatenate(Y); S=np.concatenate(S); pred=(P>=0.5).astype(int)
    # worst-site accuracy
    site_acc={int(k):accuracy_score(Y[S==k],pred[S==k]) for k in np.unique(S) if (S==k).sum()>0}
    res=dict(acc=accuracy_score(Y,pred)*100,
             bacc=balanced_accuracy_score(Y,pred)*100,
             auc=(roc_auc_score(Y,P)*100 if len(np.unique(Y))>1 else float("nan")),
             sens=recall_score(Y,pred,pos_label=1,zero_division=0)*100,
             spec=recall_score(Y,pred,pos_label=0,zero_division=0)*100,
             f1=f1_score(Y,pred,zero_division=0)*100,
             worst_site=min(site_acc.values())*100 if site_acc else float("nan"))
    return res, site_acc

res, site_acc = evaluate(clf, test_loader)
print(f"\n=== {BACKBONE.upper()} | held-out sites ===")
for k,v in res.items(): print(f"  {k:11s}: {v:.2f}")
print("  per-site acc:", {k: round(v*100,1) for k,v in site_acc.items()})
if QUICK_TEST:
    print("\n(QUICK_TEST mode — numbers are NOT meaningful, only confirm the pipeline runs.)")


## 9. t-SNE of tokens (site vs class) and loss-weight convergence

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

@torch.no_grad()
def collect_latents(tok, loader):
    tok.eval(); Z,Y,S=[],[],[]
    for b in loader:
        Z.append(tok.encoder(b["ts"].to(DEVICE)).mean(1).cpu().numpy())
        Y.append(b["label"].numpy()); S.append(b["site"].numpy())
    return np.concatenate(Z), np.concatenate(Y), np.concatenate(S)

Z,Y,S = collect_latents(tok, val_loader)
if len(Z) >= 10:
    emb=TSNE(2, perplexity=min(30,max(5,len(Z)//4)), init="pca", random_state=1).fit_transform(Z)
    fig,ax=plt.subplots(1,2,figsize=(14,5.5))
    for v in np.unique(S):
        m=S==v; ax[0].scatter(emb[m,0],emb[m,1],s=22,alpha=0.7,label=f"site {v}")
    ax[0].set_title("t-SNE colored by SITE"); ax[0].legend(fontsize=7,ncol=2)
    for c,nm in [(0,"Control"),(1,"ASD")]:
        m=Y==c; ax[1].scatter(emb[m,0],emb[m,1],s=22,alpha=0.7,label=nm)
    ax[1].set_title("t-SNE colored by CLASS"); ax[1].legend()
    for a in ax: a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.show()

# convergence of w1,w2,w3
if len(whist):
    plt.figure(figsize=(8,5))
    for j,lab in enumerate(["w1 (quant)","w2 (SigLIP)","w3 (site)"]):
        plt.plot(whist[:,j],lw=2,label=lab)
    plt.axhline(1/3,ls="--",color="gray",alpha=0.6,label="uniform 1/3")
    plt.xlabel("training step"); plt.ylabel("softmax weight"); plt.legend()
    plt.title("Loss-weight convergence"); plt.tight_layout(); plt.show()


## 10. Save checkpoints

In [ ]:
os.makedirs("/content/ckpt", exist_ok=True)
torch.save({"tokenizer":tok.state_dict(),"loss_weights":lw.state_dict()},
           f"/content/ckpt/stage1_{BACKBONE}.pt")
torch.save({"classifier":clf.state_dict()}, f"/content/ckpt/stage3_{BACKBONE}.pt")
print("saved to /content/ckpt/  — download via the Files panel (left sidebar)")
# from google.colab import files; files.download(f'/content/ckpt/stage3_{BACKBONE}.pt')


---
### Notes
- **Switch backbone:** set `BACKBONE` in cell 1 to `"gpt2"` or `"clinicalt5"` (benchmarks).
  Grok/Gemini are API models used only for the chatbot, not for this training loop.
- **Full run:** set `QUICK_TEST = False`. Expect a longer ABIDE download + training time.
- **What this adds over fMRI-LM:** a *functional* multi-class site adversary (not a
  constant-target binary one), site-balanced dynamic prototypes, and episodic Group-DRO
  for worst-site robustness — i.e. it removes the site-shift blind spot rather than
  swapping the backbone.
